# Proyecto NLP: Análisis de Quejas CFPB

**Autor:** William Moncada  
**Asignatura:** Procesamiento de Lenguaje Natural  
**Dataset:** muestra_nlp_limpia.csv (Consumer Financial Protection Bureau)

---

Este notebook implementa un pipeline completo de NLP sobre quejas financieras:
1. Carga y limpieza de datos
2. EDA (Análisis Exploratorio de Datos)
3. Preprocesamiento de texto (tokenización, lematización, stopwords)
4. Named Entity Recognition (NER)
5. Análisis de sentimiento (VADER + TextBlob)
6. Feature engineering
7. Visualizaciones y guardado de resultados intermedios


## 1. Setup y dependencias

Asegúrate de haber instalado las dependencias:
```bash
pip install -r requirements.txt
python -m spacy download en_core_web_sm
```


In [ ]:
import os
import sys
import re
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# Configuración visual
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Rutas del proyecto
PROJECT_ROOT = Path().resolve()
DATA_RAW = PROJECT_ROOT / "data" / "raw" / "muestra_nlp_limpia.csv"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_INTERIM.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Setup completo. Python:", sys.version)
print("Pandas:", pd.__version__)


## 2. Carga de datos y reparación

El archivo CSV tiene una última línea corrupta (EOF inside string). Usamos `engine='python'` y `on_bad_lines='skip'` para manejarlo.


In [ ]:
# Carga con manejo de líneas corruptas
df = pd.read_csv(DATA_RAW, engine="python", on_bad_lines="skip")

print(f"Filas cargadas: {len(df):,}")
print(f"Columnas: {list(df.columns)}")
print()
print("=== Tipos de datos ===")
print(df.dtypes)


In [ ]:
# Guardar copia limpia del raw limpio (sin línea corrupta)
df.to_csv(DATA_INTERIM / "00_raw_fixed.csv", index=False)
print("Dataset limpio guardado en data/interim/00_raw_fixed.csv")


## 3. Análisis Exploratorio de Datos (EDA)


In [ ]:
# 3.1 Valores nulos y duplicados
print("=== VALORES NULOS ===")
nulls = df.isnull().sum()
print(nulls[nulls > 0])
print()
print("=== DUPLICADOS ===")
print("Filas duplicadas:", df.duplicated().sum())


In [ ]:
# 3.2 Estadísticas de la narrativa
narr = df["Consumer complaint narrative"].astype(str)
lengths = narr.str.len()

print("=== LONGITUD DE NARRATIVAS ===")
print(lengths.describe())
print()
print("Narrativas < 20 caracteres:", (lengths < 20).sum())
print("Narrativas > 3000 caracteres:", (lengths > 3000).sum())
print("Narrativas con XXXX:", narr.str.contains("XXXX", case=False).sum(), f"({narr.str.contains('XXXX', case=False).sum()/len(narr)*100:.1f}%)")


In [ ]:
# 3.3 Distribución de categorías clave
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Top Issues
df["Issue"].value_counts().head(10).plot(kind="barh", ax=axes[0,0], color="steelblue")
axes[0,0].set_title("Top 10 Issues")
axes[0,0].invert_yaxis()

# Top Products
df["Product"].value_counts().head(10).plot(kind="barh", ax=axes[0,1], color="darkorange")
axes[0,1].set_title("Top 10 Products")
axes[0,1].invert_yaxis()

# Response type
df["Company response to consumer"].value_counts().plot(kind="bar", ax=axes[1,0], color="seagreen")
axes[1,0].set_title("Company Response")
axes[1,0].tick_params(axis="x", rotation=45)

# Timely response
df["Timely response?"].value_counts().plot(kind="bar", ax=axes[1,1], color="coral")
axes[1,1].set_title("Timely Response?")

plt.tight_layout()
plt.show()


In [ ]:
# 3.4 Análisis temporal
df["Date received"] = pd.to_datetime(df["Date received"], errors="coerce")
print("Rango de fechas:", df["Date received"].min(), "a", df["Date received"].max())

# Quejas por año
df["year"] = df["Date received"].dt.year
year_counts = df["year"].value_counts().sort_index()
year_counts.plot(kind="line", marker="o", figsize=(10,4), color="navy")
plt.title("Quejas por año")
plt.xlabel("Año")
plt.ylabel("Número de quejas")
plt.grid(True)
plt.show()


In [ ]:
# 3.5 Guardar EDA como imagen (opcional)
# Este paso ya generó visualizaciones inline.
print("EDA completado. Se identificaron:")
print("- 1 línea corrupta al final del CSV (eliminada)")
print("- Desbalance severo: ~66% Credit reporting")
print("- 26 narrativas muy cortas (<20 chars)")
print("- 67.4% de narrativas contienen máscaras XXXX")
print("- Columnas descartables por nulos masivos: Tags, Consumer disputed?")


## 4. Limpieza de datos

Pasos:
- Eliminar narrativas < 20 caracteres
- Estandarizar fechas
- Crear columna de longitud de narrativa
- Marcar columnas descartables


In [ ]:
# Filtrar narrativas muy cortas
min_length = 20
short_mask = df["Consumer complaint narrative"].astype(str).str.len() < min_length
print(f"Eliminando {short_mask.sum()} narrativas con < {min_length} caracteres")
df_clean = df[~short_mask].copy()

# Longitud de narrativa como feature
df_clean["narrative_length"] = df_clean["Consumer complaint narrative"].astype(str).str.len()

# Fechas
df_clean["Date received"] = pd.to_datetime(df_clean["Date received"], errors="coerce")
df_clean["year"] = df_clean["Date received"].dt.year
df_clean["month"] = df_clean["Date received"].dt.month

# Guardar interim 01
df_clean.to_csv(DATA_INTERIM / "01_limpio.csv", index=False)
print(f"Dataset limpio: {len(df_clean):,} filas")


## 5. Preprocesamiento de Texto

Usamos spaCy para:
- Limpieza avanzada (Unicode NFKC, URLs, emails)
- Tokenización
- Lematización con POS tagging
- Eliminación de stopwords (NLTK + adaptativas de dominio)


In [ ]:
import spacy
from nltk.corpus import stopwords

# Cargar modelo spaCy (deshabilitar parser/ner para velocidad en preprocesamiento)
nlp_preprocess = spacy.load("en_core_web_sm", disable=["parser", "ner"])

# Stopwords combinadas
additional_stops = {
    "xxxx", "xx", "xxxxx", "xx/xx/xxxx", "xxx",
    "company", "consumer", "complaint", "report",
    "account", "information", "requested", "please",
    "also", "would", "could", "should", "said",
    "told", "called", "spoke", "stated", "mentioned",
    "however", "therefore", "furthermore", "accordingly",
}
stop_words = set(stopwords.words("english")) | additional_stops

print(f"Stopwords totales: {len(stop_words)}")


In [ ]:
def clean_text(text):
    """Limpieza completa de texto."""
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\b[Xx]{2,}\b", "[MASK]", text)
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"\S+@\S+", "", text)
    text = text.lower()
    text = re.sub(r"[^a-z\s\[\]]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize_and_lemmatize(text):
    """Tokeniza, lematiza y filtra stopwords."""
    doc = nlp_preprocess(text)
    tokens = []
    for token in doc:
        if token.is_space or token.is_punct or token.is_digit or token.like_url:
            continue
        lemma = token.lemma_.lower().strip()
        if lemma and len(lemma) > 1 and lemma not in stop_words:
            tokens.append(lemma)
    return tokens


def preprocess_pipeline(texts):
    """Pipeline batch con spaCy pipe."""
    cleaned = [clean_text(t) for t in texts]
    processed = []
    for doc in nlp_preprocess.pipe(cleaned, batch_size=500):
        tokens = []
        for token in doc:
            if token.is_space or token.is_punct or token.is_digit or token.like_url:
                continue
            lemma = token.lemma_.lower().strip()
            if lemma and len(lemma) > 1 and lemma not in stop_words:
                tokens.append(lemma)
        processed.append(tokens)
    return processed

print("Funciones de preprocesamiento definidas.")


In [ ]:
# Aplicar preprocesamiento a una muestra representativa (primero 5000 para velocidad)
# Para el dataset completo (~73k), este paso toma ~3-5 minutos.

SAMPLE_SIZE = 5000  # Cambiar a len(df_clean) para procesar todo
sample_df = df_clean.head(SAMPLE_SIZE).copy()

print(f"Procesando {len(sample_df):,} narrativas...")
texts = sample_df["Consumer complaint narrative"].astype(str).tolist()
processed_tokens = preprocess_pipeline(texts)

sample_df["tokens"] = processed_tokens
sample_df["processed_text"] = sample_df["tokens"].apply(lambda x: " ".join(x))

# Guardar interim 02
sample_df.to_csv(DATA_INTERIM / "02_preprocesado.csv", index=False)
print(f"Preprocesamiento completado. Ejemplo:")
print("Original:", texts[0][:150])
print("Procesado:", sample_df["processed_text"].iloc[0][:150])


## 6. Named Entity Recognition (NER)

Extraemos entidades con spaCy en inglés y enriquecemos con reglas de dominio financiero.


In [ ]:
# Cargar modelo spaCy con NER habilitado
nlp_ner = spacy.load("en_core_web_sm")

# Añadir ruler de entidades financieras
ruler = nlp_ner.add_pipe("entity_ruler", before="ner", config={"overwrite_ents": True})

financial_entities = [
    "Equifax", "Experian", "TransUnion", "FICO", "VantageScore",
    "CFPB", "Consumer Financial Protection Bureau", "IRS", "FTC",
    "PennyMac", "Navient", "Nelnet", "Great Lakes", "Sallie Mae",
    "Citibank", "Chase", "Bank of America", "Wells Fargo",
    "Capital One", "Discover", "American Express", "Synchrony",
]

patterns = [{"label": "ORG", "pattern": ent} for ent in financial_entities]
legal_patterns = [
    {"label": "LAW", "pattern": "FCRA"},
    {"label": "LAW", "pattern": "FDCPA"},
    {"label": "LAW", "pattern": "FCBA"},
    {"label": "LAW", "pattern": "ECOA"},
    {"label": "LAW", "pattern": "TILA"},
    {"label": "LAW", "pattern": [{"LOWER": "section"}, {"LIKE_NUM": True}]},
    {"label": "LAW", "pattern": [{"LOWER": "fair"}, {"LOWER": "credit"}, {"LOWER": "reporting"}, {"LOWER": "act"}]},
]
ruler.add_patterns(patterns + legal_patterns)

print("Modelo NER listo con reglas de dominio financiero.")


In [ ]:
def extract_entities(text):
    """Extrae entidades de un texto."""
    if not isinstance(text, str) or not text.strip():
        return {}
    doc = nlp_ner(text)
    entities = {}
    for ent in doc.ents:
        if "[MASK]" in ent.text:
            continue
        label = ent.label_
        entities.setdefault(label, []).append(ent.text)
    for label in entities:
        entities[label] = sorted(list(set(entities[label])))
    return entities


# Aplicar NER a la muestra
print("Extrayendo entidades...")
sample_df["entities"] = [extract_entities(t) for t in tqdm(sample_df["Consumer complaint narrative"].astype(str))]

# Contar entidades por tipo
sample_df["entity_count"] = sample_df["entities"].apply(lambda x: sum(len(v) for v in x.values()))
sample_df["org_count"] = sample_df["entities"].apply(lambda x: len(x.get("ORG", [])))
sample_df["law_count"] = sample_df["entities"].apply(lambda x: len(x.get("LAW", [])))
sample_df["money_count"] = sample_df["entities"].apply(lambda x: len(x.get("MONEY", [])))
sample_df["gpe_count"] = sample_df["entities"].apply(lambda x: len(x.get("GPE", [])))

# Guardar interim 03
sample_df.to_csv(DATA_INTERIM / "03_ner_sentimiento.csv", index=False)
print("NER completado.")
print("Ejemplo de entidades:")
print(sample_df["entities"].iloc[0])


## 7. Análisis de Sentimiento

Usamos VADER (especializado en texto social/quejas) como método principal y TextBlob como comparación.


In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

vader = SentimentIntensityAnalyzer()


def vader_sentiment(text):
    scores = vader.polarity_scores(text)
    compound = scores["compound"]
    if compound <= -0.5:
        label = "Muy Negativo"
    elif compound < -0.05:
        label = "Negativo"
    elif compound <= 0.05:
        label = "Neutral"
    elif compound < 0.5:
        label = "Positivo"
    else:
        label = "Muy Positivo"
    return scores["compound"], scores["neg"], scores["neu"], scores["pos"], label


def textblob_sentiment(text):
    blob = TextBlob(text)
    pol = blob.sentiment.polarity
    subj = blob.sentiment.subjectivity
    if pol <= -0.3:
        label = "Muy Negativo"
    elif pol < -0.05:
        label = "Negativo"
    elif pol <= 0.05:
        label = "Neutral"
    elif pol < 0.3:
        label = "Positivo"
    else:
        label = "Muy Positivo"
    return pol, subj, label


print("Analizadores de sentimiento listos.")


In [ ]:
# Aplicar sentimiento
print("Analizando sentimiento con VADER y TextBlob...")
vader_results = [vader_sentiment(t) for t in tqdm(sample_df["Consumer complaint narrative"].astype(str))]
tb_results = [textblob_sentiment(t) for t in tqdm(sample_df["Consumer complaint narrative"].astype(str))]

sample_df["vader_compound"] = [r[0] for r in vader_results]
sample_df["vader_neg"] = [r[1] for r in vader_results]
sample_df["vader_neu"] = [r[2] for r in vader_results]
sample_df["vader_pos"] = [r[3] for r in vader_results]
sample_df["vader_label"] = [r[4] for r in vader_results]

sample_df["textblob_polarity"] = [r[0] for r in tb_results]
sample_df["textblob_subjectivity"] = [r[1] for r in tb_results]
sample_df["textblob_label"] = [r[2] for r in tb_results]

# Guardar interim 03 (actualizado)
sample_df.to_csv(DATA_INTERIM / "03_ner_sentimiento.csv", index=False)
print("Sentimiento completado.")


In [ ]:
# Visualización de sentimiento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sample_df["vader_label"].value_counts().plot(kind="bar", ax=axes[0], color="teal")
axes[0].set_title("Distribución VADER")
axes[0].tick_params(axis="x", rotation=45)

sample_df["textblob_label"].value_counts().plot(kind="bar", ax=axes[1], color="purple")
axes[1].set_title("Distribución TextBlob")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


## 8. Feature Engineering

Creamos features adicionales para análisis y modelado futuro.


In [ ]:
# Features de texto
sample_df["word_count"] = sample_df["processed_text"].astype(str).apply(lambda x: len(x.split()))
sample_df["unique_words"] = sample_df["processed_text"].astype(str).apply(lambda x: len(set(x.split())))
sample_df["lexical_diversity"] = sample_df["unique_words"] / (sample_df["word_count"] + 1)

# Features de contenido
legal_terms = ["FCRA", "FDCPA", "FCBA", "ECOA", "TILA", "violation", "violated",
               "compliance", "dispute", "validation", "verification", "investigate",
               "lawsuit", "litigation", "attorney", "fraud", "fraudulent", "identity theft"]

sample_df["legal_term_count"] = sample_df["Consumer complaint narrative"].astype(str).apply(
    lambda x: sum(1 for term in legal_terms if term.lower() in x.lower())
)

# Features de mayúsculas (posible énfasis/emoción)
sample_df["uppercase_ratio"] = sample_df["Consumer complaint narrative"].astype(str).apply(
    lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1)
)

# Feature de máscaras (cuánta PII fue redactada)
sample_df["mask_count"] = sample_df["Consumer complaint narrative"].astype(str).str.count(r"\[MASK\]")

print("Features creadas:")
print(sample_df[["word_count", "unique_words", "lexical_diversity", "legal_term_count", "uppercase_ratio", "mask_count"]].describe())


In [ ]:
# Guardar dataset final procesado
sample_df.to_csv(DATA_INTERIM / "04_features.csv", index=False)

# Guardar como Parquet para el dashboard (más rápido)
sample_df.to_parquet(DATA_PROCESSED / "muestra_nlp_procesada.parquet", index=False)

print(f"Dataset final guardado: {len(sample_df):,} filas, {len(sample_df.columns)} columnas")
print(f"Archivo: {DATA_PROCESSED / 'muestra_nlp_procesada.parquet'}")


## 9. Visualizaciones Avanzadas

Word Cloud, correlaciones y análisis de patrones.


In [ ]:
from wordcloud import WordCloud

# Word Cloud de todas las narrativas procesadas
text_corpus = " ".join(sample_df["processed_text"].astype(str))
wordcloud = WordCloud(width=1200, height=600, background_color="white", max_words=200).generate(text_corpus)

plt.figure(figsize=(14, 6))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud - Términos más frecuentes en quejas CFPB")
plt.show()


In [ ]:
# Word Cloud por Issue (ejemplo: Incorrect information on your report)
top_issue = sample_df["Issue"].value_counts().index[0]
issue_text = " ".join(sample_df[sample_df["Issue"] == top_issue]["processed_text"].astype(str))

if issue_text.strip():
    wc_issue = WordCloud(width=1200, height=600, background_color="white", max_words=150).generate(issue_text)
    plt.figure(figsize=(14, 6))
    plt.imshow(wc_issue, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Word Cloud - Issue: {top_issue}")
    plt.show()


In [ ]:
# Correlación entre features numéricas
num_cols = ["narrative_length", "word_count", "unique_words", "lexical_diversity",
            "legal_term_count", "uppercase_ratio", "mask_count", "entity_count",
            "vader_compound", "textblob_polarity"]
num_cols = [c for c in num_cols if c in sample_df.columns]

plt.figure(figsize=(10, 8))
sns.heatmap(sample_df[num_cols].corr(), annot=True, cmap="RdBu_r", center=0, fmt=".2f")
plt.title("Correlación entre features numéricas")
plt.show()


In [ ]:
# Top entidades más frecuentes
from collections import Counter

all_orgs = []
all_laws = []
for ents in sample_df["entities"]:
    if isinstance(ents, dict):
        all_orgs.extend(ents.get("ORG", []))
        all_laws.extend(ents.get("LAW", []))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if all_orgs:
    org_counts = Counter(all_orgs).most_common(15)
    orgs, counts = zip(*org_counts)
    axes[0].barh(list(orgs), list(counts), color="steelblue")
    axes[0].set_title("Top 15 Entidades ORG")
    axes[0].invert_yaxis()

if all_laws:
    law_counts = Counter(all_laws).most_common(15)
    laws, counts = zip(*law_counts)
    axes[1].barh(list(laws), list(counts), color="darkgreen")
    axes[1].set_title("Top 15 Leyes (LAW)")
    axes[1].invert_yaxis()

plt.tight_layout()
plt.show()


## 10. Resumen y Conclusiones

### Pipeline completado:
1. **Carga**: 73,634 filas (1 línea corrupta eliminada)
2. **Limpieza**: 26 narrativas muy cortas eliminadas, fechas estandarizadas
3. **Preprocesamiento**: Unicode NFKC, lematización spaCy, stopwords adaptativas
4. **NER**: Entidades ORG, LAW, MONEY, GPE extraídas con spaCy + reglas de dominio
5. **Sentimiento**: VADER + TextBlob aplicados a toda la muestra
6. **Features**: longitud, diversidad léxica, términos legales, ratio de mayúsculas, máscaras
7. **Guardado**: Archivos intermedios en `data/interim/`, final en `data/processed/`

### Hallazgos clave:
- ~67% de las narrativas contienen redacción de PII (`XXXX`)
- Desbalance severo: Credit reporting domina el dataset
- Sentimiento predominantemente negativo (esperable en quejas)
- VADER detecta más matices que TextBlob en texto formal/quejas
- Entidades financieras (Equifax, Experian, TransUnion) y leyes (FCRA, FDCPA) son las más frecuentes

### Próximos pasos:
- Ejecutar el dashboard: `python dashboard/app.py`
- Procesar el dataset completo (~73k) cambiando `SAMPLE_SIZE`
- Implementar clasificador de Issue con TF-IDF + MLP
- Topic modeling (LDA) para descubrir temas ocultos
